# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I am picking Lane 4: CTR / Engagement Opportunity Scoring

The starter pipeline already shows that a learned model can outperform a hand-written rule at identifying declining pages. But the starter label `trend_direction == "down"` is a trailing signal — it identifies pages that *have already* lost impressions. A content team with limited editorial capacity needs a different, earlier question: `which visible pages are already wasting the traffic they still have?` A page sitting at position 5 with 10,000 impressions but a CTR of 0.2% is wasting a strong slot. It has demand and placement, but something about its title, snippet, or intent match is bleeding clicks. That is an actionable gap, no time-series label required, and it translates directly to a reviewer opening the page and asking: `does this title match what searchers actually want?` The data available already has everything I need to find these pages, impressions, CTR, position, and engagement.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**1. What decision does this improve?**  
Which pages should an editor open and review for title/meta copy, intent alignment, or on-page engagement fixes — ranked by the size of their opportunity gap, not by arbitrary staleness rules.

**2. Who acts on the output, and what do they do?**  
A content editor or SEO strategist reads the ranked queue, opens the top-K pages, rewrites the title/meta description, improves snippet structure, or flags the page for an intent-match audit.

**3. What does a wrong answer cost?**  
- **False positive** (flagging a well-performing page): ~1-2 hours of editor time wasted per page reviewed.  
- **False negative** (missing a high-impression, low-CTR page): the page continues losing clicks every day it sits at a good position with a poor snippet — opportunity cost compounds over weeks.  
The asymmetry means Precision@K at a small K (top 20-50) is the right metric: the team has limited review capacity, so every queue slot must count.

**4. Why does data or ML help at all?**  
CTR expectations differ by position tier — a 0.2% CTR at position 1-3 is alarming; the same CTR at position 30+ is expected. A rule that flags `CTR < threshold` without adjusting for position generates false positives at deep positions and misses real problems at top positions. A learned scoring model (or a principled residual/gap score) can account for position, volume, content type, and intent simultaneously — the pattern is real but too multi-dimensional to write by hand.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [6]:
import pandas as pd
import numpy as np
import os

# ── Load the starter dataset ──────────────────────────────────────────────────
# Works in both Colab and locally: uses GitHub raw URL as fallback
LOCAL_PATH = "../../data/raw/content_refresh_anonymized.csv"
GITHUB_URL = "https://raw.githubusercontent.com/JamesIvanMatienzo/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(LOCAL_PATH if os.path.exists(LOCAL_PATH) else GITHUB_URL)
print(f"Rows: {len(df):,}   Columns: {df.shape[1]}   Clients: {df['client_id'].nunique()}")

# Data dictionary reminders:
# - ctr is a x100 percentage: ctr=0.76 means 0.76%, NOT 76%
# - avg_position = 0 means 'no data'; exclude for position analysis
# - trend_direction and trend_pct are label sources -- never features

# ── 1. Opportunity pool: high impressions, low CTR ───────────────────────────
# "High visibility" = impressions_90d >= 500 (good/excellent tier)
# "Low CTR"        = ctr < 0.5  (below 0.5%)
# Exclude rows with no position data (avg_position == 0)
visible = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0)]
low_ctr_visible = visible[visible["ctr"] < 0.5]

print("\n── Number 1: Opportunity Pool ───────────────────────────────")
print(f"Pages with >=500 impressions and a valid position: {len(visible):,}")
print(f"Of those, pages with CTR < 0.5%:                  {len(low_ctr_visible):,}  "
      f"({100*len(low_ctr_visible)/len(visible):.1f}% of the visible pool)")

# ── 2. CTR by position tier ───────────────────────────────────────────────────
tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
valid_tiers = [t for t in tier_order if t in visible["position_tier"].unique()]
tier_ctr = visible.groupby("position_tier")["ctr"].median().reindex(valid_tiers)

print("\n── Number 2: Median CTR (%) by Position Tier ───────────────")
for tier, val in tier_ctr.items():
    print(f"  {tier:<12}  {val:.3f}%")

if "top_3" in tier_ctr.index and "page_3_5" in tier_ctr.index:
    spread = tier_ctr["top_3"] - tier_ctr["page_3_5"]
    print(f"\n  CTR spread (top_3 vs page_3_5): {spread:.3f} percentage points.")
    print("  A flat CTR threshold would mislabel pages across tiers.")

# ── 3. Double-signal pages: low CTR AND weak engagement ──────────────────────
# "Weak engagement" = engagement_rate < 30 for pages with >=30 sessions
double_signal = low_ctr_visible[
    (low_ctr_visible["sessions_90d"] >= 30) &
    (low_ctr_visible["engagement_rate"] < 30)
]

print("\n── Number 3: Double-Signal Pages ───────────────────────────")
print(f"Low-CTR visible pages ALSO with weak engagement (rate<30%, >=30 sessions):")
print(f"  Count:              {len(double_signal):,}")
print(f"  Median impressions: {double_signal['impressions_90d'].median():,.0f}")
print(f"  Median CTR:         {double_signal['ctr'].median():.3f}%")
print(f"  Median eng. rate:   {double_signal['engagement_rate'].median():.1f}%")
print("  These pages bleed clicks at the result AND fail to retain visitors.")

Rows: 30,000   Columns: 44   Clients: 32

── Number 1: Opportunity Pool ───────────────────────────────
Pages with >=500 impressions and a valid position: 16,726
Of those, pages with CTR < 0.5%:                  14,245  (85.2% of the visible pool)

── Number 2: Median CTR (%) by Position Tier ───────────────
  top_3         0.200%
  page_1        0.240%
  striking      0.170%
  page_3_5      0.090%
  deep          0.000%

  CTR spread (top_3 vs page_3_5): 0.110 percentage points.
  A flat CTR threshold would mislabel pages across tiers.

── Number 3: Double-Signal Pages ───────────────────────────
Low-CTR visible pages ALSO with weak engagement (rate<30%, >=30 sessions):
  Count:              5,133
  Median impressions: 9,281
  Median CTR:         0.180%
  Median eng. rate:   1.6%
  These pages bleed clicks at the result AND fail to retain visitors.


### What those numbers tell me

**Number 1 — The opportunity pool is large (14,245 pages, 85.2% of visible):**  
Of the 16,726 pages with ≥500 impressions and a valid position, 85.2% have CTR below 0.5%. That is far too long a list to act on with a flat rule — which is exactly what a position-adjusted gap score is for: it narrows the list to where the gap between observed and expected CTR is largest.

**Number 2 — Position tier matters but isn't simple (top_3: 0.200%, page_1: 0.240%, striking: 0.170%, page_3_5: 0.090%, deep: 0.000%):**  
Notably, `top_3` median CTR (0.200%) is actually *lower* than `page_1` (0.240%) in this dataset. This suggests top-3 slots may be dominated by high-impression competitive queries where SERP features (ads, featured snippets) depress organic CTR, even at top positions. **A flat CTR threshold cannot capture this complexity — position-tier adjustment is the analytical core of Lane 4.**

**Number 3 — 5,133 double-signal pages with median 9,281 impressions and 1.6% engagement rate:**  
These pages are simultaneously low-CTR *and* show near-zero post-click engagement. They bleed clicks at the search result *and* fail to retain visitors. This double-signal pattern cannot be identified from any single metric — which is why a multi-signal approach earns its place here and why this lane is worth seven weeks.

## 4. Careful words: what I can and can't claim
*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What this work CAN say

- **Observed:** "In the 30,000-row starter dataset, 85.2% of pages with ≥500 impressions have CTR below 0.5%, with meaningful variation across position tiers."
- **Measured:** "The position-adjusted CTR gap score ranked pages such that X of the top 20 were confirmed as review candidates by hand inspection."
- **Directional:** "5,133 pages show both low CTR and weak engagement (median engagement rate: 1.6%) — a segment a single-metric rule cannot identify, suggesting a multi-signal approach adds value."
- **Decision-support:** "This ranked queue helps a content team prioritise which pages to open and review first. It does not automate the editorial decision."

### What this work CANNOT say

- **No causation:** "Rewriting the title of page X will increase its CTR." — This data is observational. We watched what happened; we did not run an experiment.
- **No algorithm claims:** "We discovered a Google ranking factor." — We observed correlations between content signals and CTR. Google's algorithm is not in our data.
- **No future prediction from a snapshot:** "This model predicts future CTR." — The starter dataset is a single 90-day snapshot. The model captures the current pattern; it does not guarantee future outcomes.
- **No single-cause diagnosis:** "Low CTR always means the title is bad." — Low CTR can also reflect intent mismatch, SERP feature competition, seasonality, or low query volume. The model surfaces candidates for review, not diagnoses.
- **No generalisation claims:** "This result generalises to all clients." — The dataset covers 32 pseudonymised clients. Results are directional evidence, not universal benchmarks.

## Self-check

Before you submit, confirm each line honestly:

- [/] Every section above is filled — markdown thinking AND the code that backs it
- [/] The notebook runs top to bottom with no errors (Runtime → Run all)
- [/] No client names, URLs, or private queries anywhere
- [/] My claims use careful words: observed, measured, directional, decision-support
- [/] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.